# Tool-Call Fine-Tune Lab — QLoRA Pipeline

Fine-tunes **Qwen2.5-7B-Instruct** on a curated tool-calling dataset using QLoRA (4-bit NF4) on a Kaggle T4 GPU.

**Dataset:** `doeonkim00/tool-call-finetune-data` (attached as Kaggle dataset input)

**Expected runtime:** ~8–10 hours on T4 (16 GB VRAM)

**Stack:** `transformers`, `peft`, `datasets`, `bitsandbytes` — all pre-installed on Kaggle GPU kernels.

Internet should remain enabled so the notebook can download the base model when no cached Kaggle model input is attached. No additional pip installs are required.

In [ ]:
# Cell 0 — Install / upgrade missing packages
import subprocess, sys

packages = ["bitsandbytes>=0.46.1", "peft>=0.13.0", "accelerate>=1.0.0"]
for pkg in packages:
    print(f"Installing {pkg}...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-U", pkg, "-q"],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"  FAILED: {result.stderr.strip()[:200]}")
    else:
        print(f"  OK")

# Verify bitsandbytes
try:
    import bitsandbytes as bnb
    print(f"\nbitsandbytes version: {bnb.__version__}")
except ImportError as e:
    print(f"\nFATAL: bitsandbytes not available: {e}")
    raise

In [ ]:
# Cell 2 — Imports + config

import os
import json
import torch
from pathlib import Path

# ---------------------------------------------------------------------------
# Kaggle Secrets (WANDB_API_KEY and HF_TOKEN are optional)
# ---------------------------------------------------------------------------
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    try:
        WANDB_API_KEY = _secrets.get_secret("WANDB_API_KEY")
    except Exception:
        WANDB_API_KEY = None
    try:
        HF_TOKEN = _secrets.get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
except ImportError:
    WANDB_API_KEY = os.environ.get("WANDB_API_KEY")
    HF_TOKEN = os.environ.get("HF_TOKEN")

print(f"W&B key present: {bool(WANDB_API_KEY)}")
print(f"HF token present: {bool(HF_TOKEN)}")

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
DATA_DIR   = Path("/kaggle/input/tool-call-finetune-data")
OUTPUT_DIR = Path("/kaggle/working")
LORA_DIR   = OUTPUT_DIR / "qwen25-7b-tool-call-lora"
MERGED_DIR = OUTPUT_DIR / "qwen25-7b-tool-call-merged"
RESULTS_DIR = OUTPUT_DIR / "results"

for d in [LORA_DIR, MERGED_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Model path: scan ALL /kaggle/input/ for config.json to find the model
# ---------------------------------------------------------------------------
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
HF_MODEL_ID = BASE_MODEL  # Keep original for merge step

kaggle_input = Path("/kaggle/input")
if kaggle_input.exists():
    print(f"Scanning {kaggle_input} for Qwen model...")
    for p in sorted(kaggle_input.rglob("config.json")):
        try:
            cfg = json.loads(p.read_text())
            if "qwen2" in cfg.get("model_type", "").lower() or "Qwen" in cfg.get("_name_or_path", ""):
                BASE_MODEL = str(p.parent)
                print(f"Found Qwen model at: {BASE_MODEL}")
                break
        except Exception:
            continue
    else:
        print(f"No cached model found. Will download: {BASE_MODEL}")
        # List what IS available for debugging
        for child in kaggle_input.iterdir():
            print(f"  /kaggle/input/{child.name}/")
            if child.is_dir():
                for sub in child.iterdir():
                    print(f"    {sub.name}/")

print(f"\nUsing model: {BASE_MODEL}")

# ---------------------------------------------------------------------------
# Training hyperparameters (T4-optimised)
# ---------------------------------------------------------------------------
MAX_SEQ_LENGTH  = 2048
EPOCHS          = 1
LR              = 2e-4
BATCH_SIZE      = 1
GRAD_ACCUM      = 8
WARMUP_RATIO    = 0.1
LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.05

print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"CUDA available:       {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:                  {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 3 — Load data from Kaggle dataset path

def load_jsonl(path: Path):
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]

train_data = load_jsonl(DATA_DIR / "train.jsonl")
val_data   = load_jsonl(DATA_DIR / "val.jsonl")
test_data  = load_jsonl(DATA_DIR / "test.jsonl")

print(f"Train: {len(train_data):,} examples")
print(f"Val:   {len(val_data):,} examples")
print(f"Test:  {len(test_data):,} examples")

sample = train_data[0]
print(f"\nSample source:   {sample.get('source', 'n/a')}")
print(f"Sample category: {sample.get('category', 'n/a')}")
msgs = sample.get('messages', [])
if len(msgs) > 1:
    print(f"User msg:        {msgs[1]['content'][:120]}")
if len(msgs) > 2:
    print(f"Tool calls:      {len(msgs[2].get('tool_calls', []))}")

In [ ]:
# Cell 4 — Configure T4-optimized training (no trl dependency)

from transformers import BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig

# 4-bit NF4 quantisation — fp16 for T4 (no bf16 support)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

training_args = TrainingArguments(
    output_dir=str(LORA_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    fp16=True,
    bf16=False,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="wandb" if WANDB_API_KEY else "none",
    run_name="qwen25-7b-tool-call-lora-t4",
    remove_unused_columns=False,
)

print("BnB config:")
print(f"  quant_type={bnb_config.bnb_4bit_quant_type}, compute_dtype=fp16, double_quant=True")
print("LoRA config:")
print(f"  r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print("Training args:")
print(f"  epochs={EPOCHS}, batch={BATCH_SIZE}, grad_accum={GRAD_ACCUM}, lr={LR}")
print(f"  fp16=True, gradient_checkpointing=True")

In [ ]:
# Cell 5 — Load Qwen2.5-7B with 4-bit quantization + LoRA

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import get_peft_model, prepare_model_for_kbit_training

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    padding_side="right",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print(f"\nGPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

In [ ]:
# Cell 6 — Format dataset for SFT training (ChatML format with tool_call markup)

from datasets import Dataset

def format_for_training(example):
    """Convert JSONL record to a single ChatML string for SFTTrainer."""
    messages = example.get("messages", [])
    tools    = example.get("tools", [])

    # Build tool description appended to the system message
    tool_desc = ""
    if tools:
        tool_desc = "\n\nAvailable tools:"
        for tool in tools:
            fn = tool.get("function", {})
            tool_desc += f"\n- {fn.get('name', '')}: {fn.get('description', '')}"

    parts = []
    for msg in messages:
        role    = msg["role"]
        content = msg.get("content") or ""

        if role == "system":
            parts.append(f"<|im_start|>system\n{content}{tool_desc}<|im_end|>")

        elif role == "user":
            parts.append(f"<|im_start|>user\n{content}<|im_end|>")

        elif role == "tool":
            # Tool result back to the model
            parts.append(f"<|im_start|>tool\n{content}<|im_end|>")

        elif role == "assistant":
            tool_calls = msg.get("tool_calls", [])
            if tool_calls:
                tc_strs = []
                for tc in tool_calls:
                    fn   = tc.get("function", {})
                    args = fn.get("arguments", {})
                    if isinstance(args, str):
                        try:
                            args = json.loads(args)
                        except json.JSONDecodeError:
                            pass
                    tc_strs.append(json.dumps({"name": fn.get("name", ""), "arguments": args}))
                assistant_content = "<tool_call>\n" + "\n".join(tc_strs) + "\n</tool_call>"
            else:
                assistant_content = content
            parts.append(f"<|im_start|>assistant\n{assistant_content}<|im_end|>")

    return {"text": "\n".join(parts)}


train_dataset = Dataset.from_list(train_data).map(format_for_training, desc="Formatting train")
val_dataset   = Dataset.from_list(val_data).map(format_for_training,   desc="Formatting val")

# Drop examples that exceed the context window
train_dataset = train_dataset.filter(
    lambda x: len(tokenizer.encode(x["text"])) <= MAX_SEQ_LENGTH,
    desc="Filtering train by length",
)
val_dataset = val_dataset.filter(
    lambda x: len(tokenizer.encode(x["text"])) <= MAX_SEQ_LENGTH,
    desc="Filtering val by length",
)

print(f"Train after length filter: {len(train_dataset):,}")
print(f"Val after length filter:   {len(val_dataset):,}")
print(f"\nSample formatted text (first 500 chars):")
print(train_dataset[0]["text"][:500])

In [ ]:
# Cell 7 — Train with transformers.Trainer (no trl needed)

from transformers import Trainer, DataCollatorForLanguageModeling

if WANDB_API_KEY:
    import wandb
    wandb.login(key=WANDB_API_KEY)
    os.environ["WANDB_PROJECT"] = "tool-call-finetune-lab"
    print("W&B logged in")
else:
    os.environ["WANDB_DISABLED"] = "true"
    print("W&B disabled (no WANDB_API_KEY secret)")

# Tokenize the formatted text
def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )

print("Tokenizing datasets...")
train_tokenized = train_dataset.map(tokenize_fn, batched=True, remove_columns=train_dataset.column_names, desc="Tokenizing train")
val_tokenized = val_dataset.map(tokenize_fn, batched=True, remove_columns=val_dataset.column_names, desc="Tokenizing val")

# Add labels (same as input_ids for causal LM)
def add_labels(examples):
    examples["labels"] = examples["input_ids"].copy()
    return examples

train_tokenized = train_tokenized.map(add_labels, desc="Adding labels train")
val_tokenized = val_tokenized.map(add_labels, desc="Adding labels val")

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
)

steps_per_epoch = len(train_tokenized) // (BATCH_SIZE * GRAD_ACCUM)
print(f"Training {len(train_tokenized):,} examples for {EPOCHS} epoch(s)")
print(f"Steps per epoch: {steps_per_epoch}")
print(f"GPU memory before train: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

trainer.train()
trainer.save_model(str(LORA_DIR))
tokenizer.save_pretrained(str(LORA_DIR))

print(f"Training complete. LoRA adapter saved to {LORA_DIR}")

In [ ]:
# Cell 8 — Merge LoRA adapter into base model

from peft import PeftModel

# For merging we need the original HF model name (not Kaggle path)
MERGE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct" if "/kaggle/" in BASE_MODEL else BASE_MODEL

print("Loading base model for merging (on CPU to preserve VRAM)...")
base_model_for_merge = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,  # Use Kaggle cache if available
    torch_dtype=torch.float16,
    device_map="cpu",
    trust_remote_code=True,
)

print("Attaching LoRA adapter...")
peft_model = PeftModel.from_pretrained(base_model_for_merge, str(LORA_DIR))

print("Merging weights...")
merged_model = peft_model.merge_and_unload()

merged_model.save_pretrained(str(MERGED_DIR))
tokenizer.save_pretrained(str(MERGED_DIR))

print(f"Merged model saved to {MERGED_DIR}")

# Free memory before eval
del base_model_for_merge, peft_model
torch.cuda.empty_cache()

In [ ]:
# Cell 9 — Quick eval: test a few tool-call prompts

from transformers import pipeline as hf_pipeline

print("Loading merged model for inference...")
pipe = hf_pipeline(
    "text-generation",
    model=str(MERGED_DIR),
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
    device_map="auto",
)

quick_prompts = [
    (
        "get_weather",
        "<|im_start|>system\nYou are a helpful assistant.\n\nAvailable tools:\n- get_weather: Get current weather for a location<|im_end|>\n"
        "<|im_start|>user\nWhat's the weather in Tokyo?<|im_end|>\n<|im_start|>assistant\n",
    ),
    (
        "search_web",
        "<|im_start|>system\nYou are a helpful assistant.\n\nAvailable tools:\n- search_web: Search the internet<|im_end|>\n"
        "<|im_start|>user\nFind the latest news about AI.<|im_end|>\n<|im_start|>assistant\n",
    ),
]

for expected_fn, prompt in quick_prompts:
    result = pipe(prompt, max_new_tokens=200, do_sample=False)
    generated = result[0]["generated_text"][len(prompt):]
    hit = expected_fn in generated
    print(f"\n[{'PASS' if hit else 'FAIL'}] Expected function: {expected_fn}")
    print(f"Output: {generated[:300]}")

In [ ]:
# Cell 10 — BFCL eval on test set (first 100 examples for speed)

bfcl_test = [ex for ex in test_data if ex.get("source") == "bfcl"]
print(f"BFCL test examples total: {len(bfcl_test)}")

correct = 0
total   = 0
results_by_category = {}

for ex in bfcl_test[:100]:
    category = ex.get("category", "unknown")
    results_by_category.setdefault(category, {"correct": 0, "total": 0})

    # Build prompt (strip the final assistant turn so model generates it)
    formatted = format_for_training(ex)
    prompt    = formatted["text"].rsplit("<|im_start|>assistant", 1)[0] + "<|im_start|>assistant\n"

    output    = pipe(prompt, max_new_tokens=300, do_sample=False)
    generated = output[0]["generated_text"][len(prompt):]

    # Expected function names from ground-truth tool_calls
    expected_calls = ex["messages"][2].get("tool_calls", []) if len(ex["messages"]) > 2 else []
    expected_names = {tc["function"]["name"] for tc in expected_calls}

    # Loose match: all expected function names appear in the generated text
    generated_names = {name for name in expected_names if name in generated}
    is_correct      = generated_names == expected_names

    if is_correct:
        correct += 1
        results_by_category[category]["correct"] += 1
    total += 1
    results_by_category[category]["total"] += 1

overall_acc = correct / total * 100 if total > 0 else 0.0

print(f"\n{'='*60}")
print(f"BFCL Eval Results (first 100 examples)")
print(f"{'='*60}")
print(f"Overall: {correct}/{total}  ({overall_acc:.1f}%)")
print("\nPer category:")
for cat, stats in sorted(results_by_category.items()):
    acc = stats["correct"] / stats["total"] * 100 if stats["total"] > 0 else 0
    print(f"  {cat:30s}  {stats['correct']}/{stats['total']}  ({acc:.1f}%)")

In [ ]:
# Cell 11 — Save results + push to HuggingFace (if HF_TOKEN available)

# Save BFCL eval results locally
eval_results = {
    "model": f"{BASE_MODEL} + QLoRA",
    "overall_accuracy": f"{overall_acc:.1f}%",
    "correct": correct,
    "total": total,
    "categories": {
        k: {"correct": v["correct"], "total": v["total"],
            "accuracy": f"{v['correct']/v['total']*100:.1f}%" if v["total"] > 0 else "n/a"}
        for k, v in results_by_category.items()
    },
    "training": {
        "epochs": EPOCHS,
        "train_examples": len(train_dataset),
        "val_examples":   len(val_dataset),
        "gpu": "T4 16GB (Kaggle)",
        "fp16": True,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
    },
}

results_path = RESULTS_DIR / "bfcl_results.json"
results_path.write_text(json.dumps(eval_results, indent=2))
print(f"Results saved to {results_path}")

# Push LoRA adapter to HuggingFace Hub
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    hf_repo_owner = os.environ.get("HF_REPO_OWNER", "doeonkim00")
    repo_id = f"{hf_repo_owner}/qwen2.5-7b-tool-calling-lora"
    api.create_repo(repo_id, exist_ok=True, private=False)
    api.upload_folder(
        folder_path=str(LORA_DIR),
        repo_id=repo_id,
        commit_message=f"QLoRA adapter — BFCL acc {overall_acc:.1f}%",
    )
    print(f"LoRA adapter pushed to https://huggingface.co/{repo_id}")
else:
    print("HF_TOKEN not set — skipping HuggingFace push.")
    print("Add HF_TOKEN to Kaggle Secrets to enable automatic upload.")

## Summary

| Step | Detail |
|------|--------|
| Base model | Qwen2.5-7B-Instruct |
| Quantisation | 4-bit NF4 (QLoRA) |
| LoRA rank | 16, alpha 32 |
| Training | 1 epoch, fp16, batch 1 × grad_accum 8 |
| Hardware | Kaggle T4 16 GB |
| Dataset | `doeonkim00/tool-call-finetune-data` |
| Eval | BFCL (first 100 test examples, loose function-name match) |

**Next steps**
- Run full BFCL eval (all test examples, strict argument matching)
- AWQ quantisation for faster serving
- vLLM deployment + stage-pilot bridge test
- 3-epoch run once baseline is confirmed